# Final Task B 17-class - MobileNetV3-Large

Train MobileNetV3-Large directly on the final 17 species classes. This uses ImageNet pretrained initialization and does not load the old 20-class Task B checkpoints.


In [2]:
from pathlib import Path
import sys

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    ROOT = next((parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / 'src').exists()), Path.cwd())

sys.path.insert(0, str(ROOT))

import os
os.chdir(ROOT)
FINAL_B_DIR = ROOT / "notebooks" / "phase 1" / "task-B-17-new"
sys.path.insert(0, str(FINAL_B_DIR))
print(f"ROOT: {ROOT}")
print(f"task-B-17-new: {FINAL_B_DIR}")


ROOT: /workspace/woodbench
task-B-17-new: /workspace/woodbench/notebooks/phase 1/task-B-17-new


In [3]:
# Colab-only setup. Cloud data is expected to be already converted to 17 classes
# by 00_data_preparation.ipynb. This download only runs when the dataset is absent.
if Path('/content').exists():
    !pip install -q -r {ROOT}/requirements.txt

    DATA_DIR = ROOT / "data" / "task-B-group-split"
    if not DATA_DIR.exists():
        from huggingface_hub import login, snapshot_download
        import os

        token = os.getenv("HF_TOKEN", "")
        if token:
            login(token=token)
        snapshot_download(
            repo_id="ny0g507h3p/wood-task-B",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
            ignore_patterns=["*.md", "README*"],
        )


In [4]:
from final_b_17class import (
    BACKBONE_CONFIGS,
    prepare_environment,
    run_all_experiments,
    summarize_results,
    verify_17class_dataset,
)

prepare_environment(ROOT)

BACKBONE = "mobilenetv3_large"
DISPLAY_NAME = "MobileNetV3-Large"
DATA_ROOT = ROOT / "data" / "task-B-group-split"
BATCH_SIZE = BACKBONE_CONFIGS[BACKBONE]["batch_size"]
EPOCHS = 50

print(f"Backbone: {DISPLAY_NAME} ({BACKBONE})")
print(f"Batch size: {BATCH_SIZE}")
print(f"Data root: {DATA_ROOT}")


Backbone: MobileNetV3-Large (mobilenetv3_large)
Batch size: 64
Data root: /workspace/woodbench/data/task-B-group-split


In [5]:
class_names = verify_17class_dataset(DATA_ROOT)
print(f"Classes ({len(class_names)}):")
for idx, class_name in enumerate(class_names):
    print(f"  {idx:02d}: {class_name}")


[Data] train: 17 classes, 3365 images
[Data] val: 17 classes, 903 images
[Data] test: 17 classes, 532 images
Classes (17):
  00: Araucaria araucana
  01: Callitris columellaris
  02: Calocedrus rupestris
  03: Chamaecyparis obtusa
  04: Cryptomeria japonica
  05: Cunninghamia lanceolata
  06: Fokienia hodginsii
  07: Picea glauca
  08: Pinus cernua
  09: Pinus dalatensis
  10: Pinus krempfii
  11: Pinus radiata
  12: Podocarpus pilgeri
  13: Pseudotsuga brevifolia
  14: Taxus wallichiana
  15: Thujopsis dolabrata
  16: Xanthocyparis vietnamensis


In [6]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")

results = run_all_experiments(
    backbone_name=BACKBONE,
    root=ROOT,
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    device=DEVICE,
)


GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GB
[Data] train: 17 classes, 3365 images
[Data] val: 17 classes, 903 images
[Data] test: 17 classes, 532 images
[Device] cuda
[Output] /workspace/woodbench/data/experiments/Task-B-new/final

main/seed_42/mobilenetv3_large
[main | mobilenetv3_large s42] train=3365, val=903, test=532, classes=17


model.safetensors:   0%|          | 0.00/22.1M [00:00<?, ?B/s]

Epoch 1/50 | Train Loss: 2.8540, Acc: 5.53% | Val Loss: 2.8488, Acc: 5.54% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 5.54%)


Epoch 2/50 | Train Loss: 2.7528, Acc: 17.43% | Val Loss: 2.6631, Acc: 25.03% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 25.03%)


Epoch 3/50 | Train Loss: 2.4490, Acc: 49.61% | Val Loss: 2.2924, Acc: 56.59% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 56.59%)


Epoch 4/50 | Train Loss: 1.9209, Acc: 71.24% | Val Loss: 1.7094, Acc: 72.43% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 72.43%)


... (180 outputs left out)

Epoch 45/50 | Train Loss: 0.6445, Acc: 99.49% | Val Loss: 0.6836, Acc: 96.90% | LR: 3.02e-06[EarlyStop 8/10]


Epoch 46/50 | Train Loss: 0.6443, Acc: 99.28% | Val Loss: 0.6816, Acc: 97.12% | LR: 1.94e-06[EarlyStop 9/10]


Epoch 47/50 | Train Loss: 0.6444, Acc: 99.43% | Val Loss: 0.6847, Acc: 96.90% | LR: 1.09e-06[EarlyStop 10/10]

[EarlyStopping] No improvement for 10 epochs. Stopping at epoch 47.
[ERROR] taskB_90rotation/seed_42/mobilenetv3_large: RuntimeError('[enforce fail at inline_container.cc:747] . open file failed with strerror: No space left on device')

taskB_label_smoothing_0.0/seed_42/mobilenetv3_large
[ERROR] taskB_label_smoothing_0.0/seed_42/mobilenetv3_large: OSError(28, 'No space left on device')

taskB_label_smoothing_0.2/seed_42/mobilenetv3_large
[ERROR] taskB_label_smoothing_0.2/seed_42/mobilenetv3_large: OSError(28, 'No space left on device')


RuntimeError: mobilenetv3_large failed runs: {'taskB_90rotation/seed_42/mobilenetv3_large': "RuntimeError('[enforce fail at inline_container.cc:747] . open file failed with strerror: No space left on device')", 'taskB_label_smoothing_0.0/seed_42/mobilenetv3_large': "OSError(28, 'No space left on device')", 'taskB_label_smoothing_0.2/seed_42/mobilenetv3_large': "OSError(28, 'No space left on device')"}

In [ ]:
summarize_results(results)


## Continue after disk-full interruption

Run the root, import/config, and data-check cells above, but do not rerun the old `run_all_experiments` cell. The 90-degree rotation run reached early stopping at epoch 47 and only the final `last.pth` save failed, so finalize it from the valid best checkpoint. Use `run_remaining` for the two unstarted label-smoothing runs. Run one model notebook at a time.

In [7]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")


GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GB


In [9]:
from final_b_17class import (
    EXPERIMENTS,
    experiment_output_dir,
    load_completed_results,
    train_one,
)

continued_results = {
    (row["experiment"], row["seed"]): row
    for row in load_completed_results(backbone_name=BACKBONE, root=ROOT)
}

def run_remaining(experiment_name, seed):
    result = train_one(
        root=ROOT,
        backbone=BACKBONE,
        seed=seed,
        experiment_name=experiment_name,
        experiment_cfg=EXPERIMENTS[experiment_name],
        data_root=DATA_ROOT,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        device=DEVICE,
        resume_training=True,
    )
    continued_results[(experiment_name, seed)] = result
    return result

def finalize_from_best(experiment_name, seed):
    output_dir = experiment_output_dir(ROOT, experiment_name, seed, BACKBONE)
    best_path = output_dir / f"{BACKBONE}_seed{seed}_best.pth"
    if not best_path.exists():
        raise FileNotFoundError(f"Missing best checkpoint: {best_path}")
    result = train_one(
        root=ROOT,
        backbone=BACKBONE,
        seed=seed,
        experiment_name=experiment_name,
        experiment_cfg=EXPERIMENTS[experiment_name],
        data_root=DATA_ROOT,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        device=DEVICE,
        resume_training=False,
    )
    continued_results[(experiment_name, seed)] = result
    return result

print(f"Loaded {len(continued_results)}/6 completed configs for {BACKBONE}.")


Loaded 3/6 completed configs for mobilenetv3_large.


In [10]:
# Main seed 456 is already complete; this only confirms/reuses its metrics.
run_remaining("main", 456)


[main | mobilenetv3_large s456] Reusing /workspace/woodbench/data/experiments/Task-B-new/final/main/seed_456/mobilenetv3_large/test_metrics.json


{'backbone': 'mobilenetv3_large',
 'seed': 456,
 'experiment': 'main',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 94.57364341085271,
 'best_epoch': 28,
 'final_train_acc': 99.42908653846153,
 'final_val_acc': 93.7984496124031,
 'stopped_early': True,
 'epochs_trained': 38,
 'test_accuracy': 0.981203007518797,
 'test_f1_macro': 0.9816077086992405,
 'test_precision_macro': 0.9832098379892498,
 'test_recall_macro': 0.9811620160884866,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 1.0,
  'Calocedrus rupestris': 1.0,
  'Chamaecyparis obtusa': 1.

In [11]:
# Training reached early stopping at epoch 47; only evaluate/register best.pth.
finalize_from_best("taskB_90rotation", 42)


[taskB_90rotation | mobilenetv3_large s42] Found checkpoint: /workspace/woodbench/data/experiments/Task-B-new/final/taskB_90rotation/mobilenetv3_large/mobilenetv3_large_seed42_best.pth
[taskB_90rotation | mobilenetv3_large s42] test_acc=98.68% F1=98.79%


{'backbone': 'mobilenetv3_large',
 'seed': 42,
 'experiment': 'taskB_90rotation',
 'num_classes': 17,
 'best_val_acc': 97.23145071982282,
 'best_epoch': 37,
 'final_train_acc': 99.12860576923077,
 'final_val_acc': 97.23145071982282,
 'stopped_early': False,
 'epochs_trained': 37,
 'test_accuracy': 0.9868421052631579,
 'test_f1_macro': 0.9878722877650359,
 'test_precision_macro': 0.9884858120152238,
 'test_recall_macro': 0.9881535947712418,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 1.0,
  'Calocedrus rupestris': 1.0,
  'Chamaecyparis obtusa': 1.0,
  'Cryptomeria japonica': 1.0,
  'Cunninghamia lanceolata': 1.0,
  'Fokienia hodginsii': 1.0,
  'Picea glauca': 1.0,
  'Pinus cernua': 1.0,
  'Pinus dalatensis': 0.927536231884058,
  'Pinus krempfii': 0.8985507246376812,
  'Pinus radiata': 1.0,
  'Podocarpus pilgeri': 1.0,
  'Pseudotsuga brevifolia': 1.0,
  'Taxus wallichiana': 1.0,
  'Thujopsis dolabrata': 1.0,
  'Xanthocyparis vietnamensis': 0.967741935483

In [12]:
# Remaining label-smoothing=0.0 ablation
run_remaining("taskB_label_smoothing_0.0", 42)


[taskB_label_smoothing_0.0 | mobilenetv3_large s42] train=3365, val=903, test=532, classes=17


Epoch 1/50 | Train Loss: 2.8542, Acc: 5.53% | Val Loss: 2.8481, Acc: 5.54% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 5.54%)


Epoch 2/50 | Train Loss: 2.7415, Acc: 17.46% | Val Loss: 2.6409, Acc: 24.47% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 24.47%)


Epoch 3/50 | Train Loss: 2.3993, Acc: 49.28% | Val Loss: 2.2192, Acc: 56.48% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 56.48%)


Epoch 4/50 | Train Loss: 1.7823, Acc: 70.76% | Val Loss: 1.5179, Acc: 71.87% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 71.87%)


Epoch 5/50 | Train Loss: 1.0098, Acc: 84.22% | Val Loss: 0.8748, Acc: 81.40% | LR: 1.00e-04[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 81.40%)


... (24 outputs left out)

Epoch 30/50 | Train Loss: 0.0331, Acc: 99.19% | Val Loss: 0.1723, Acc: 93.36% | LR: 4.13e-05[EarlyStop 9/10]


Epoch 31/50 | Train Loss: 0.0313, Acc: 99.25% | Val Loss: 0.1862, Acc: 93.13% | LR: 3.79e-05[EarlyStop 10/10]

[EarlyStopping] No improvement for 10 epochs. Stopping at epoch 31.


[taskB_label_smoothing_0.0 | mobilenetv3_large s42] val_acc=93.91% test_acc=97.93% F1=98.07%


{'backbone': 'mobilenetv3_large',
 'seed': 42,
 'experiment': 'taskB_label_smoothing_0.0',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 93.9091915836102,
 'best_epoch': 21,
 'final_train_acc': 99.24879807692308,
 'final_val_acc': 93.13399778516057,
 'stopped_early': True,
 'epochs_trained': 31,
 'test_accuracy': 0.9793233082706767,
 'test_f1_macro': 0.9807270358681649,
 'test_precision_macro': 0.982000389986185,
 'test_recall_macro': 0.980784942182001,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 1.0,
  'Calocedrus rupestris': 1.0,
  'Chamae

In [13]:
# Remaining label-smoothing=0.2 ablation
run_remaining("taskB_label_smoothing_0.2", 42)


[taskB_label_smoothing_0.2 | mobilenetv3_large s42] train=3365, val=903, test=532, classes=17


Epoch 1/50 | Train Loss: 2.8539, Acc: 5.53% | Val Loss: 2.8494, Acc: 5.54% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 5.54%)


Epoch 2/50 | Train Loss: 2.7642, Acc: 17.46% | Val Loss: 2.6854, Acc: 25.25% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 25.25%)


Epoch 3/50 | Train Loss: 2.4985, Acc: 49.76% | Val Loss: 2.3653, Acc: 56.37% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 56.37%)


Epoch 4/50 | Train Loss: 2.0563, Acc: 71.69% | Val Loss: 1.8933, Acc: 72.98% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 72.98%)


Epoch 5/50 | Train Loss: 1.5913, Acc: 85.37% | Val Loss: 1.5512, Acc: 82.95% | LR: 1.00e-04[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 82.95%)


... (43 outputs left out)

Epoch 49/50 | Train Loss: 1.0785, Acc: 99.37% | Val Loss: 1.1683, Acc: 94.24% | LR: 1.22e-07[EarlyStop 6/10]


Epoch 50/50 | Train Loss: 1.0776, Acc: 99.37% | Val Loss: 1.1659, Acc: 94.57% | LR: 0.00e+00[EarlyStop 7/10]


[taskB_label_smoothing_0.2 | mobilenetv3_large s42] val_acc=94.68% test_acc=98.68% F1=98.79%


{'backbone': 'mobilenetv3_large',
 'seed': 42,
 'experiment': 'taskB_label_smoothing_0.2',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 94.6843853820598,
 'best_epoch': 43,
 'final_train_acc': 99.36899038461539,
 'final_val_acc': 94.57364341085271,
 'stopped_early': False,
 'epochs_trained': 50,
 'test_accuracy': 0.9868421052631579,
 'test_f1_macro': 0.9878722877650359,
 'test_precision_macro': 0.9884858120152238,
 'test_recall_macro': 0.9881535947712418,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 1.0,
  'Calocedrus rupestris': 1.0,
  'Cha

In [14]:
results = list(continued_results.values())
summarize_results(results)



Task B 17-class summary
experiment                       seed   val_acc  test_acc        f1      prec       rec
----------------------------------------------------------------------------------------
main                               42    94.91%    98.68%    98.79%    98.85%    98.82%
main                              123    95.57%    98.50%    98.58%    98.70%    98.61%
main                              456    94.57%    98.12%    98.16%    98.32%    98.12%
taskB_90rotation                   42    97.23%    98.68%    98.79%    98.85%    98.82%
taskB_label_smoothing_0.0          42    93.91%    97.93%    98.07%    98.20%    98.08%
taskB_label_smoothing_0.2          42    94.68%    98.68%    98.79%    98.85%    98.82%
----------------------------------------------------------------------------------------
main test_acc: 98.43 +/- 0.23%
main test_f1:  98.51 +/- 0.26%
